# Data preparation and quality checks

## tl;dr
Cleaned **558,129 transactions**, **115,204 cost records**, and **112 gate assignments**.
Removed 2,887 blank rows and 1,856 repeated transaction records. There are **54 directional routes**
and **68 unique gate leases**. **7,902 originally ambiguous transaction dates are interpreted as
day/month/year (DD/MM/YYYY)**, as requested. Original date strings and assumption flags are retained.

## Context & Methods
Reusable cleaning lives in [`src/clean_data.py`](../src/clean_data.py).
Run `python3 src/clean_data.py` from the project root to regenerate the outputs.
This notebook records checks and conclusions; the six question notebooks remain the home for
business calculations, visualizations and recommendations.

### Key Assumptions
- Local `data/raw/csv/` exports are the inputs; raw Excel files are unchanged.
- Field definitions and airport codes come from the supplied data dictionary and case package.
- Route direction matters. Missing membership does not mean Non-Member.
- Duplicate IDs are consolidated only when cleaned business fields and possible dates agree.
- Both day-first and month-first slash dates occur. For the remaining 7,902 ambiguous dates,
  assume DD/MM/YYYY (day first), as requested. This is an analysis assumption, not verified source metadata.
  Unambiguous dates and dates resolved using agreeing duplicates retain their existing interpretation.
- Under this assumption, 759 transactions (net CAD 499,209.97) fall outside the stated
  September 2024–August 2026 study period. Keep them in the cleaned file and reconcile them
  separately from the two fiscal years; do not silently include them in either year.
- Amounts are CAD; refunds stay negative. No monetary scaling, outlier removal or imputation.
- Gate leases use calendar years; the end date is exclusive. No missing leases are forecast here.

## Data
Quality checks and SHA-256 source hashes are computed in memory from the raw CSVs.
All code cells use the Python standard library and can run from the project or notebook directory.


In [1]:
import csv
import sys
from decimal import Decimal
from itertools import islice
from pathlib import Path

project = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src/clean_data.py").exists())
processed = project / "data/processed"
if str(project) not in sys.path:
    sys.path.insert(0, str(project))

from src.clean_data import prepare_data

tables, report = prepare_data(project / "data/raw/csv")
del tables

for name, source in report["sources"].items():
    print(f"{name}: data/raw/csv/{source['file']} | SHA-256 {source['sha256']}")


transactions: data/raw/csv/transactions.csv | SHA-256 d236eae5f249cc3459cb74aaf38fa958a716787a62437f54b4f258b177066238
costs: data/raw/csv/costs.csv | SHA-256 06af3187e74e81672ebff44a3bc69b0e356aa23baec47d8bfab62d34b8ed9754
gates: data/raw/csv/gates.csv | SHA-256 95de2dc4b85d8d067bb1bcf07bc995895aae5184ef0eb80613169ab071bfd6c8


## Results
### Row reconciliation
Blank rows are spreadsheet padding. Repeated transaction IDs are ingestion duplicates,
including records that differ only in formatting. Distinct IDs with identical amounts remain separate.


In [2]:
print(f"{'Dataset':<15} {'Input':>10} {'Blank':>8} {'Duplicates':>11} {'Output':>10}")

for name, stats in report["tables"].items():
    assert stats["input_rows"] == stats["blank_rows_removed"] + stats["duplicate_rows_removed"] + stats["output_rows"]
    assert Decimal(stats["input_amount_total"]) == Decimal(stats["output_amount_total"]) + Decimal(stats["duplicate_amount_removed"])

    print(f"{name:<15} {stats['input_rows']:>10,} {stats['blank_rows_removed']:>8,} {stats['duplicate_rows_removed']:>11,} {stats['output_rows']:>10,}")

print("\nTransaction CAD removed with duplicate records:", report["tables"]["transactions"]["duplicate_amount_removed"])
print("Cleaned net transaction CAD:", report["tables"]["transactions"]["output_amount_total"])
print("Operating cost CAD:", report["tables"]["costs"]["output_amount_total"])


Dataset              Input    Blank  Duplicates     Output
transactions       560,985    1,000       1,856    558,129
costs              116,204    1,000           0    115,204
gates                  999      887           0        112

Transaction CAD removed with duplicate records: 1230798.17
Cleaned net transaction CAD: 370629024.85
Operating cost CAD: 431362040.24


### Dates and missing values
The 7,902 dates with two valid interpretations now use the requested day-first assumption
(DD/MM/YYYY). For example, `10/02/2025` is February 10, 2025. Their cleaned timestamp is populated,
with `date_status = assumed_day_first` and the original text in `transaction_date_raw`.
Monthly, fiscal-year and other date-based findings must disclose this assumption.
An additional 57 dates were resolved using agreeing duplicates.
Missing membership and payment values remain explicit gaps, rather than fabricated categories.


In [3]:
transactions = report["tables"]["transactions"]

for status, count in sorted(transactions["date_status"].items()):
    print(f"{status}: {count:,} ({count / transactions['output_rows']:.2%})")

for field in ["membership_tier", "payment_method"]:
    count = transactions["missing_values"][field]
    print(f"Missing {field}: {count:,} ({count / transactions['output_rows']:.2%})")

with (processed / "transactions.csv").open(newline="") as stream:
    examples = list(islice((r for r in csv.DictReader(stream) if r["date_status"] == "ambiguous"), 3))

print("\nDates assigned using the day-first assumption:")
for row in examples:
    print(row["transaction_id"], row["transaction_date_raw"], "->", row["transaction_date"], row["date_status"])


assumed_day_first: 7,902 (1.42%)
parsed: 550,170 (98.57%)
resolved_duplicate: 57 (0.01%)
Missing membership_tier: 8,355 (1.50%)
Missing payment_method: 4,443 (0.80%)



Dates assigned using the day-first assumption:


### Gate costs and route coverage
`gates.csv` retains the route assignments, including four records without a route.
The separate `gate_leases.csv` counts the lease cost once per airport/gate/year.
All transaction routes have cost coverage, and all cost routes have a gate assignment.
Route coverage does not establish a transaction-to-flight join.


In [4]:
with (processed / "gate_leases.csv").open(newline="") as stream:
    leases = list(csv.DictReader(stream))

assert len({(r["city"], r["gate_number"], r["year"]) for r in leases}) == len(leases)

print(f"Unique gate leases: {len(leases)}")
print("Unassigned gate-years:", sum(int(r["assigned_route_count"]) == 0 for r in leases))

for year, expected in report["gate_leases"]["annual_cost_by_lease_year"].items():
    total = sum((Decimal(r["annual_gate_cost"]) for r in leases if r["year"] == year), Decimal(0))

    assert total == Decimal(expected)

    print(f"{year}: CAD {total:,.2f}")

for check, routes in report["route_coverage"].items():
    assert not routes, (check, routes)

    print(f"{check}: {len(routes)}")


Unique gate leases: 68
Unassigned gate-years: 4
2024-2025: CAD 9,664,400.00
2025-2026: CAD 9,877,700.00
transactions_without_cost_route: 0
cost_routes_without_gate_assignment: 0


## Takeaways
- Use the processed CSVs for the question notebooks. Cleaning is repeatable in `src/clean_data.py`.
- Deduplication removes CAD 1,230,798.17 of repeated signed transaction amounts; operating costs are unchanged.
- Disclose the DD/MM/YYYY assumption for 7,902 originally ambiguous dates in time-based findings.
  Use the assumed dates for time-based grouping. The 759 transactions outside the study period
  are retained in the data and separately reconciled in fiscal-year analyses. Missing membership
  (8,355 records) and payment method (4,443 records) also limit segment analysis.
- Sum gate expenses from `gate_leases.csv`. Annual totals are CAD 9,664,400.00 for 2024-2025
  and CAD 9,877,700.00 for 2025-2026. Forecasting the missing lease year belongs in notebook 05.
- Purchase dates and flight dates represent different events. This preparation does not infer service dates,
  passenger connections, fiscal-year allocations or route-level gate allocations.


### Study-period reconciliation
The requested day-first convention places some dates outside the stated case horizon. This is a consequence of the assumption, not an additional deletion. Show their record count and signed CAD amount beside fiscal-year results.

In [5]:
from src.analysis_common import load_data

scope_data = load_data("transactions")

scope_summary = scope_data.groupby("fiscal_year")["amount_cents"].agg(transactions="count", net_amount_cents="sum")
scope_summary["net_revenue_cad"] = scope_summary["net_amount_cents"] / 100

assert scope_summary.loc["Outside study period", "transactions"] == 759
assert scope_summary.loc["Outside study period", "net_amount_cents"] == 49_920_997

display(scope_summary[["transactions", "net_revenue_cad"]])

del scope_data

,transactions,net_revenue_cad
fiscal_year,,
FY2024-25,269811,1.804194e+08
FY2025-26,287559,1.897104e+08
Outside study period,759,4.992100e+05
